# Equity News Pack（米国株版）

PostgreSQL 上のニュースデータから、LLM（OpenAI 互換 API）で市場インプリケーションのある記事を抽出・要約し、blpapi で銘柄エンリッチを行ってローカル Parquet に蓄積するノートブック。最終アウトプットは ``US_All.csv``。

## 処理フロー
1. **データ取得**: PostgreSQL から指定期間のニュースを取得（トピックフィルタ + 言語）
2. **機械的フィルタ**: AI 生成ニュース除外、RIC → BBG Ticker 変換、指数構成銘柄に絞り込み
3. **LLM 処理**: 市場インプリケーション判定 → 本文要約・センチメント分類 → 日次アブストラクト生成
4. **保存**: ローカル Parquet にアップサート（``data/US_accumulated.parquet``）→ ``US_All.csv`` を書き出し

地域固有のパラメータは ``src/region_config.py`` の ``US_CONFIG`` を参照。

## 1. 基本設定

In [ ]:
# autoreload を有効化（src/ 編集時に kernel 再起動なしで反映）
ip = get_ipython()
if "IPython.extensions.autoreload" not in ip.extension_manager.loaded:
    ip.run_line_magic("load_ext", "autoreload")
ip.run_line_magic("autoreload", "2")

In [ ]:
import polars as pl

from src.enrichment import (
    add_date_from_capture_time,
    aggregate_by_suid,
    explode_related_rics,
    fetch_index_universe_tickers,
    fetch_sectors_for_tickers,
    filter_ai_generated_news,
    filter_to_index_universe,
)
from src.llm_analyzer import (
    LLMAnalyzer,
    add_summary_and_sentiment,
    filter_by_market_implication,
    generate_daily_abstracts,
)
from src.news_loader import (
    deduplicate_by_clean_headline,
    query_news,
)
from src.region_config import US_CONFIG
from src.settings import Settings
from src.storage import (
    serialize_list_columns_to_string,
    upsert_parquet_local,
    write_all_tsv,
)

In [ ]:
# --- 取得期間 ---
START_DATE = "2026-01-05"
END_DATE = "2026-01-09"

# --- 地域別パラメータ（トピック / 指数 / RIC マップ 等） ---
region = US_CONFIG
print(region)

# --- 接続設定（.env から） ---
settings = Settings.from_env()
print(f"DB host: {settings.db.host}")
print(f"LLM model: {settings.llm.model}")

# --- 蓄積データの保存先 ---
ACCUM_PARQUET = settings.data_dir / f"{region.region_code}_accumulated.parquet"
ALL_CSV = f"{region.region_code}_All.csv"

## 2. ニュースデータの取得

In [ ]:
# PostgreSQL から指定期間のニュースを取得
# - トピックフィルタ（OR 条件）と除外トピックは region_config の設定を使用
# - 言語フィルタ ("ja" / "en") も自動適用
df_raw = query_news(
    db=settings.db,
    start_date=START_DATE,
    end_date=END_DATE,
    language=region.language,
    include_topics=region.individual_topics,
    exclude_topics=region.exclude_topics,
)
print(f"取得件数: {df_raw.height}")
print(df_raw.head())

In [ ]:
# プロバイダ違いの同一記事を統合（"RTRS:" 等のプレフィクスを除いた本文部分で重複削除）
df_dedup = deduplicate_by_clean_headline(df_raw)
print(f"重複削除後: {df_raw.height} -> {df_dedup.height} 件")

## 3. 機械的フィルタリング

In [ ]:
# AI 生成ニュース（Bloomberg Automation 配信）を除外
before = df_dedup.height
df_filtered = filter_ai_generated_news(df_dedup, region.ai_generated_prefix_pattern)
print(f"AI 生成除去: {before} -> {df_filtered.height} 件 ({before - df_filtered.height} 件除外)")

In [ ]:
# related_rics（スペース区切り）を行展開し、地域マップで RIC -> BBG Ticker 変換
# 地域外の RIC（例: JP 処理時の AAPL.O）は変換失敗で除外される
df_exploded = explode_related_rics(df_filtered, region.ric_suffix_to_bbg_label)
print(f"RIC 展開後: {df_exploded.height} 行（BBGID 単位）")
print(df_exploded.select(["SUID", "BBG_Ticker"]).head())

In [ ]:
# blpapi セッション開始 + 指数構成銘柄取得 + Sector 一括取得
import blpapi

session_opts = blpapi.SessionOptions()
session_opts.setServerHost(settings.blpapi.host)
session_opts.setServerPort(settings.blpapi.port)
bbg_session = blpapi.Session(session_opts)
assert bbg_session.start(), "blpapi セッション起動失敗（Bloomberg Terminal 起動中？）"
assert bbg_session.openService("//blp/refdata"), "//blp/refdata 接続失敗"

# 指数構成銘柄を取得
index_universe = fetch_index_universe_tickers(
    session=bbg_session,
    index_ticker=region.index_ticker,
    reference_date=region.index_reference_date,
    bbg_exchange_codes=region.bbg_index_exchange_codes,
    canonical_label=region.canonical_exchange_label,
)
print(f"{region.index_ticker} 構成銘柄: {index_universe.height} 銘柄")

# 構成銘柄の Sector も先に取得しておく（後で left join で付与）
sector_map = fetch_sectors_for_tickers(bbg_session, index_universe["BBG_Ticker"].to_list())
sector_df = pl.DataFrame({
    "BBG_Ticker": list(sector_map.keys()),
    "Sector": list(sector_map.values()),
})
print(f"Sector 取得済み銘柄: {sector_df.filter(pl.col('Sector').is_not_null()).height} / {sector_df.height}")

In [ ]:
# 指数構成銘柄に該当しないニュースを除外
df_universe = filter_to_index_universe(df_exploded, index_universe)
print(f"指数構成銘柄に該当: {df_exploded.height} -> {df_universe.height} 行")

# Sector を付与
df_with_sector = df_universe.join(sector_df, on="BBG_Ticker", how="left")
print(df_with_sector.select(["SUID", "BBG_Ticker", "Sector"]).head())

## 4. LLM による絞り込み・要約処理

以下 3 タスクを実行する。

- **Task 1**: 市場インプリケーションのある記事のみ抽出
- **Task 2**: 本文要約 + 市場センチメント分類（-1 / 0 / +1）
- **Task 3**: 日付単位で要約をまとめ、日次アブストラクトを生成

In [ ]:
# LLM クライアントの初期化
analyzer = LLMAnalyzer(settings.llm)

In [ ]:
# Task 1: 市場インプリケーションありのみ抽出
print("Task 1: Filtering by market implication...")
selected_df = filter_by_market_implication(df_with_sector, analyzer)
print(f"Selected {selected_df.height} rows from {df_with_sector.height}.")
print(selected_df.head())

In [ ]:
# 同一 SUID で複数銘柄に展開された行を 1 行に集約
# （BBG_Ticker と Sector はリスト化、それ以外は first）
selected_df = aggregate_by_suid(selected_df, list_cols=["BBG_Ticker", "Sector"])
print(f"SUID 集約後: {selected_df.height} 件")
print(selected_df.select(["SUID", "BBG_Ticker", "Sector", "Headline"]).head())

In [ ]:
# Task 2: 本文要約 + 市場センチメント分類
print("Task 2: Summarizing Body text and classifying sentiment...")
sum_df = add_summary_and_sentiment(selected_df, analyzer)
sum_df = add_date_from_capture_time(sum_df)
print(sum_df.select(["SUID", "Summarized_Text", "Market_Sentiment"]).head())

In [ ]:
# Task 3: 日次アブストラクトを生成（参考出力）
print("Task 3: Generating Daily Abstracts...")
abstract_df = generate_daily_abstracts(sum_df, analyzer)
print(abstract_df)

## 5. 蓄積保存と CSV 出力

In [ ]:
# 出力スキーマに整える（List 型カラムはカンマ結合してから書き出す）
output_cols = [
    "SUID",
    "BBG_Ticker",
    "date",
    "Headline",
    "Summarized_Text",
    "Sector",
    "Market_Sentiment",
]
output_df = (
    serialize_list_columns_to_string(sum_df, ["BBG_Ticker", "Sector"])
    .select(output_cols)
)
print(output_df)

In [ ]:
# ローカル Parquet にアップサート（既存 + 今回の処理結果 → SUID で重複排除）
merged = upsert_parquet_local(ACCUM_PARQUET, output_df)
print(f"蓄積データ: {merged.height} 件 (path={ACCUM_PARQUET})")

# US_All.csv を書き出し（TSV + 行番号インデックス、参照ファイル仕様準拠）
write_all_tsv(merged.sort("date"), ALL_CSV)
print(f"CSV 出力: {ALL_CSV} ({merged.height} 行)")

In [ ]:
# blpapi セッションを閉じる
bbg_session.stop()